# Global Oil Supply/Demand Balance Model

**Last Updated:** March 2026  
**Data Sources:** EIA STEO, OPEC MOMR, IEA OMR, EIA Weekly Petroleum Status Report  
**Methodology:** Multi-source aggregation with EIA STEO as structural backbone, overlaid with OPEC/IEA data where available

---

In [ ]:
# === Setup ===
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
from pathlib import Path
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")

from src.eia_client import EIAClient
from src.steo_loader import STEOLoader
from src.opec_loader import OPECLoader
from src.iea_loader import IEALoader
from src.balance_model import OilBalanceModel
from src.charts import (
    plot_supply_vs_demand,
    plot_balance_bar,
    plot_implied_vs_actual_inventory,
    plot_opec_production_stacked,
    plot_supply_breakdown,
    plot_demand_by_region,
    plot_call_on_opec,
)
from src.commentary import CommentaryGenerator
from src.styling import apply_professional_style, style_balance_table, COLORS

apply_professional_style()
print("Setup complete.")

In [ ]:
# === Configuration ===
# Everything is auto-detected from the STEO workbook — no manual date updates needed.

import json

load_dotenv("../.env")
EIA_API_KEY = os.getenv("EIA_API_KEY", "")

START_DATE = "2023-01"
END_DATE = "2026-12"

# Paths
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(exist_ok=True)
(OUTPUT_DIR / "charts").mkdir(exist_ok=True)

---
## 1. Data Ingestion

In [ ]:
# === 1a. EIA STEO (Primary Baseline) ===
# Always parse from the workbook to get fresh metadata.
# Download is auto-skipped if cached workbook is < 7 days old.

steo = STEOLoader(cache_dir="../data/raw/steo")
steo_path = steo.download()
steo_data = steo.parse(steo_path, start=START_DATE, end=END_DATE)

# Auto-detect last actual month from STEO metadata
LAST_ACTUAL_MONTH = steo.metadata["last_historical_month_str"]

print(f"STEO Edition:          {steo.metadata['forecast_month']}")
print(f"Last Historical Month: {LAST_ACTUAL_MONTH}  (auto-detected)")
print(f"Data range:            {steo_data.index.min().strftime('%Y-%m')} to {steo_data.index.max().strftime('%Y-%m')}")
print(f"Columns:               {len(steo_data.columns)}")

In [ ]:
# === 1b. OPEC Production Data ===
opec = OPECLoader()
opec_prod_path = "../data/templates/opec_production_template.csv"

if opec.has_data(opec_prod_path):
    opec_prod = opec.load_production_csv(opec_prod_path)
    print(f"OPEC country-level data loaded: {opec_prod.dropna(how='all').shape[0]} months")
else:
    opec_prod = None
    print("No OPEC country data — using STEO aggregate for OPEC production.")

In [ ]:
# === 1c. IEA OMR Data ===
iea = IEALoader()
iea_supply = iea.load_supply_csv("../data/templates/iea_supply_template.csv")
iea_demand = iea.load_demand_csv("../data/templates/iea_demand_template.csv")

has_iea_supply = iea.has_data(iea_supply)
has_iea_demand = iea.has_data(iea_demand)
print(f"IEA supply data: {'Available' if has_iea_supply else 'Not entered — using STEO'}")
print(f"IEA demand data: {'Available' if has_iea_demand else 'Not entered — using STEO'}")

In [ ]:
# === 1d. US Crude Inventories (Cross-Check) ===
# Always fetch fresh from EIA API if key is configured.
# Falls back to cache if API is unavailable.

if EIA_API_KEY and EIA_API_KEY != "your_api_key_here":
    print("Fetching US crude inventories from EIA API...")
    eia = EIAClient(api_key=EIA_API_KEY)
    us_stocks = eia.get_monthly_crude_stocks(start=START_DATE)
    print(f"Fetched: {len(us_stocks)} months ({us_stocks.index.min().strftime('%Y-%m')} to {us_stocks.index.max().strftime('%Y-%m')})")
else:
    us_stocks_cache = PROCESSED_DIR / "us_crude_stocks.csv"
    if us_stocks_cache.exists():
        us_stocks = pd.read_csv(us_stocks_cache, index_col="month", parse_dates=True).squeeze()
        print(f"Loaded cached US stocks: {len(us_stocks)} months")
    else:
        us_stocks = pd.Series(dtype=float, name="us_crude_stocks_mmbbl")
        print("No EIA API key and no cache — US inventory cross-check unavailable.")

---
## 2. Build the Balance Model

In [ ]:
# === Build Model ===
model = OilBalanceModel()

# Supply
model.build_supply_table(
    steo_data=steo_data,
    opec_data=opec_prod,
    iea_data=iea_supply if has_iea_supply else None,
)
print(f"Supply table: {model.supply.shape[0]} months, {model.supply.shape[1]} columns")

# Demand
model.build_demand_table(
    steo_data=steo_data,
    iea_data=iea_demand if has_iea_demand else None,
)
print(f"Demand table: {model.demand.shape[0]} months, {model.demand.shape[1]} columns")

# Balance
balance = model.calculate_balance()
print(f"Balance calculated: {balance.shape[0]} months")

# Inventory comparison
if not us_stocks.empty:
    model.add_actual_inventory_comparison(us_stocks)
    print("US inventory comparison added.")

# Flag actuals vs forecasts (auto-detected from STEO metadata)
model.flag_actuals_vs_forecasts(LAST_ACTUAL_MONTH)
print(f"Data flagged: actuals through {LAST_ACTUAL_MONTH} (auto-detected), forecasts after.")

---
## 3. Key Metrics Dashboard

In [ ]:
# === Key Metrics Summary Cards ===
# Show the latest month's headline numbers at a glance

latest_data = model.get_latest_month_data()
stock_chg = latest_data['stock_change']

metrics_html = f"""
<div style="display: flex; gap: 16px; margin: 20px 0; flex-wrap: wrap;">
    <div style="background: #f8f9fa; padding: 20px; border-radius: 8px; flex: 1; min-width: 150px; text-align: center; border-left: 4px solid {COLORS['supply']};">
        <div style="font-size: 12px; color: #6c757d; text-transform: uppercase; letter-spacing: 1px;">World Supply</div>
        <div style="font-size: 28px; font-weight: bold; color: #2c3e50; margin: 8px 0;">{latest_data['total_supply']:.1f}</div>
        <div style="font-size: 12px; color: #6c757d;">mb/d</div>
    </div>
    <div style="background: #f8f9fa; padding: 20px; border-radius: 8px; flex: 1; min-width: 150px; text-align: center; border-left: 4px solid {COLORS['demand']};">
        <div style="font-size: 12px; color: #6c757d; text-transform: uppercase; letter-spacing: 1px;">World Demand</div>
        <div style="font-size: 28px; font-weight: bold; color: #2c3e50; margin: 8px 0;">{latest_data['total_demand']:.1f}</div>
        <div style="font-size: 12px; color: #6c757d;">mb/d</div>
    </div>
    <div style="background: {'#d5f5e3' if stock_chg > 0 else '#fadbd8'}; padding: 20px; border-radius: 8px; flex: 1; min-width: 150px; text-align: center; border-left: 4px solid {'#27ae60' if stock_chg > 0 else '#e74c3c'};">
        <div style="font-size: 12px; color: #6c757d; text-transform: uppercase; letter-spacing: 1px;">Balance</div>
        <div style="font-size: 28px; font-weight: bold; color: {'#27ae60' if stock_chg > 0 else '#e74c3c'}; margin: 8px 0;">
            {'+' if stock_chg > 0 else ''}{stock_chg:.2f}
        </div>
        <div style="font-size: 12px; color: #6c757d;">mb/d ({'surplus' if stock_chg > 0 else 'deficit'})</div>
    </div>
</div>
<div style="display: flex; gap: 16px; margin: 0 0 20px 0; flex-wrap: wrap;">
    <div style="background: #f8f9fa; padding: 16px; border-radius: 8px; flex: 1; min-width: 150px; text-align: center; border-left: 4px solid {COLORS['opec']};">
        <div style="font-size: 11px; color: #6c757d; text-transform: uppercase;">OPEC Crude</div>
        <div style="font-size: 22px; font-weight: bold; color: #2c3e50; margin: 6px 0;">{latest_data.get('opec_crude', 0):.1f}</div>
        <div style="font-size: 11px; color: #6c757d;">mb/d</div>
    </div>
    <div style="background: #f8f9fa; padding: 16px; border-radius: 8px; flex: 1; min-width: 150px; text-align: center; border-left: 4px solid {COLORS['non_opec']};">
        <div style="font-size: 11px; color: #6c757d; text-transform: uppercase;">Non-OPEC Supply</div>
        <div style="font-size: 22px; font-weight: bold; color: #2c3e50; margin: 6px 0;">{latest_data.get('non_opec', 0):.1f}</div>
        <div style="font-size: 11px; color: #6c757d;">mb/d</div>
    </div>
    <div style="background: #f8f9fa; padding: 16px; border-radius: 8px; flex: 1; min-width: 150px; text-align: center; border-left: 4px solid #8e44ad;">
        <div style="font-size: 11px; color: #6c757d; text-transform: uppercase;">Monthly Implied</div>
        <div style="font-size: 22px; font-weight: bold; color: #2c3e50; margin: 6px 0;">{stock_chg * 30:.0f}</div>
        <div style="font-size: 11px; color: #6c757d;">mmb stock {'build' if stock_chg > 0 else 'draw'}</div>
    </div>
</div>
<p style="color: #6c757d; font-size: 11px;">Period: {latest_data['report_month']}</p>
"""
display(HTML(metrics_html))

---
## 4. Global Oil Supply/Demand Balance Table

The centerpiece of the model. All values in million barrels per day (mb/d) unless noted.  
**Yellow rows** = EIA STEO forecasts. All others = published actuals/estimates.

In [ ]:
# === THE BALANCE TABLE ===
master = model.get_master_table()

# Separate display columns from metadata
meta_cols = ["Market State", "Data Type"]
display_cols = [c for c in master.columns if c not in meta_cols]

# Format month index for display
display_df = master[display_cols].copy()
display_df.index = display_df.index.strftime("%Y-%m")

# Style the table
def highlight_balance(val):
    """Color-code the balance column."""
    if isinstance(val, (int, float)) and not np.isnan(val):
        if val > 0.1:
            return "color: #27ae60; font-weight: bold"
        elif val < -0.1:
            return "color: #e74c3c; font-weight: bold"
    return ""

def highlight_forecast_rows(row):
    """Yellow background for forecast rows."""
    month_str = row.name
    if month_str > LAST_ACTUAL_MONTH[:7]:
        return ["background-color: #fff3cd"] * len(row)
    return [""] * len(row)

styled = (
    display_df.style
    .format("{:.2f}", na_rep="—")
    .set_caption("Global Oil Supply/Demand Balance (mb/d)")
    .set_table_styles([
        {"selector": "caption", "props": [
            ("font-size", "16px"), ("font-weight", "bold"),
            ("text-align", "left"), ("padding-bottom", "12px"),
            ("color", "#2c3e50"),
        ]},
        {"selector": "th", "props": [
            ("background-color", "#2c3e50"), ("color", "white"),
            ("font-weight", "bold"), ("text-align", "center"),
            ("padding", "8px 10px"), ("font-size", "11px"),
            ("border-bottom", "2px solid #1a252f"),
        ]},
        {"selector": "td", "props": [
            ("text-align", "right"), ("padding", "5px 8px"),
            ("font-size", "11px"), ("border-bottom", "1px solid #ecf0f1"),
        ]},
        {"selector": "th.row_heading", "props": [
            ("background-color", "#34495e"), ("color", "white"),
            ("font-weight", "bold"), ("text-align", "left"),
            ("padding", "5px 10px"),
        ]},
    ])
    .apply(highlight_forecast_rows, axis=1)
)

# Apply balance highlighting to the balance column if it exists
if "Balance (mb/d)" in display_df.columns:
    styled = styled.map(highlight_balance, subset=["Balance (mb/d)"])

display(styled)

# Export
master.to_excel(OUTPUT_DIR / "tables" / "oil_balance_table.xlsx")
master.to_csv(OUTPUT_DIR / "tables" / "oil_balance_table.csv")
print(f"\nExported to {OUTPUT_DIR / 'tables'}")

---
## 5. Quarterly Summary

In [ ]:
# === Quarterly Averages ===
quarterly = model.get_quarterly_summary()

q_styled = (
    quarterly.style
    .format("{:.2f}")
    .set_caption("Quarterly Oil Balance (mb/d, average)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"), ("padding", "8px")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "5px 8px")]},
    ])
)

display(q_styled)

---
## 6. Charts

In [ ]:
# === Chart 1: Global Supply vs Demand ===
last_actual = pd.to_datetime(LAST_ACTUAL_MONTH)

fig = plot_supply_vs_demand(
    model.balance,
    last_actual_month=last_actual,
    save_path=str(OUTPUT_DIR / "charts" / "supply_vs_demand.png"),
)
plt.show()

print(f"[Actuals through {LAST_ACTUAL_MONTH}, forecasts shaded]")

In [ ]:
# === Chart 2: Implied Stock Change (Balance Bar) ===
fig = plot_balance_bar(
    model.balance,
    last_actual_month=last_actual,
    save_path=str(OUTPUT_DIR / "charts" / "balance_bar.png"),
)
plt.show()

In [ ]:
# === Chart 3: Implied vs Actual US Inventory ===
fig = plot_implied_vs_actual_inventory(
    model.balance,
    last_actual_month=last_actual,
    save_path=str(OUTPUT_DIR / "charts" / "implied_vs_actual.png"),
)
plt.show()

In [ ]:
# === Chart 4: OPEC Production by Country ===
if opec_prod is not None and not opec_prod.dropna(how='all').empty:
    fig = plot_opec_production_stacked(
        opec_prod,
        save_path=str(OUTPUT_DIR / "charts" / "opec_stacked.png"),
    )
    plt.show()
else:
    print("OPEC country-level data not available. Fill in opec_production_template.csv to enable this chart.")

In [ ]:
# === Chart 5: Supply Breakdown (Stacked Area) ===
fig = plot_supply_breakdown(
    model.supply,
    last_actual_month=last_actual,
    save_path=str(OUTPUT_DIR / "charts" / "supply_breakdown.png"),
)
plt.show()

In [ ]:
# === Chart 6: Demand by Region (Stacked Area) ===
fig = plot_demand_by_region(
    model.demand,
    last_actual_month=last_actual,
    save_path=str(OUTPUT_DIR / "charts" / "demand_by_region.png"),
)
plt.show()

In [ ]:
# === Chart 7: Call on OPEC vs Actual Production ===
fig = plot_call_on_opec(
    model.balance,
    model.supply,
    last_actual_month=last_actual,
    save_path=str(OUTPUT_DIR / "charts" / "call_on_opec.png"),
)
plt.show()

---
## 7. Market Commentary

In [ ]:
# === Auto-Generated Commentary ===
# Edit this output to add your own views and market color

cg = CommentaryGenerator()
commentary = cg.generate_for_latest_month(model)
display(Markdown(commentary))

---
## 8. Term Structure Cross-Check

Compare balance implications with Brent curve shape:
- **Surplus** (builds) → expect contango (M1 < M6)
- **Deficit** (draws) → expect backwardation (M1 > M6)

In [ ]:
# === Term Structure Validation ===
from src.futures_loader import FuturesLoader

futures = FuturesLoader()
brent_path = "../data/templates/brent_futures_template.csv"

if futures.has_data(brent_path):
    brent = futures.load_csv(brent_path)
    print("Brent futures curve data loaded.")
    display(brent.tail(6))
else:
    # Try fetching front-month from Yahoo Finance
    brent_front = futures.fetch_brent_front_month(start="2023-01-01")
    if not brent_front.empty:
        print("Brent M1 from Yahoo Finance:")
        display(brent_front.tail(6))
    else:
        print("No Brent futures data available.")
        print(f"Fill in {brent_path} with Bloomberg/Reuters data for curve analysis.")

# Summarize implied curve shape from balance
recent_balance = model.balance["implied_stock_change"].tail(3).mean()
if recent_balance > 0.1:
    implied_shape = "contango (surplus → builds → downward-sloping curve)"
elif recent_balance < -0.1:
    implied_shape = "backwardation (deficit → draws → upward-sloping near-term)"
else:
    implied_shape = "flat (balanced market)"

print(f"\nRecent 3-month avg balance: {recent_balance:+.2f} mb/d")
print(f"Implied curve shape: {implied_shape}")

---
## 9. Scenario Analysis

The EIA STEO is one view of the future. Desks build their own assumptions on top. Below we test
how different OPEC production and demand outcomes change the balance.

Adjustments apply to **forecast months only** (after the last actual). Actuals are never touched.

---
## Data Sources & Methodology

| Source | Coverage | Access | Update Frequency |
|--------|----------|--------|------------------|
| EIA STEO | Global S/D balance + forecasts | Free | Monthly (~10th) |
| OPEC MOMR | OPEC production by country | Free (PDF/Excel) | Monthly (~12th) |
| IEA OMR | Non-OPEC supply, demand by region | Paywalled (headlines free) | Monthly (~15th) |
| EIA Weekly | US crude inventories | Free API | Weekly (Wed) |

**Methodology:** EIA STEO provides the structural backbone (complete global balance with 18-month forecasts). 
OPEC MOMR and IEA OMR data are used as overrides where they offer better granularity or more timely estimates. 
Balance = Total Supply − Total Demand = Implied Stock Change (mb/d). 
Positive balance = surplus/build. Negative balance = deficit/draw.

**Note:** Yellow-highlighted rows indicate EIA STEO forecast projections. All other rows are published actuals/estimates.

---

### Dynamic Data Flow
This notebook is **fully dynamic** — no manual date updates needed:
- **STEO workbook**: Auto-downloaded (cached for 7 days, then refreshed)
- **Last Actual Month**: Auto-detected from STEO `Dates` sheet metadata
- **US Inventories**: Fetched live from EIA API each run
- **Actual/Forecast flagging**: Driven by auto-detected metadata

**To refresh:** Just click Kernel → Restart & Run All. The notebook handles everything.

In [ ]:
# === Define Scenarios ===
from src.scenario import ScenarioEngine

engine = ScenarioEngine(steo_data, last_actual_month=LAST_ACTUAL_MONTH)

# The STEO forecasts OPEC dropping from 29.25 to 23.08 mb/d in one month (Mar 2026).
# That implies a massive production unwind. Most market participants expect something
# less dramatic, so testing alternatives is critical.

engine.add_scenario(
    "OPEC Holds Cuts",
    opec_crude_adj=+4.0,
    description="OPEC maintains ~27 mb/d instead of cutting to 23. Most probable alternative.",
)

engine.add_scenario(
    "China Demand Weakness",
    demand_adj=-0.4,
    description="Chinese economic slowdown drags global demand by 400 kb/d.",
)

engine.add_scenario(
    "Bull Case (Tight Market)",
    opec_crude_adj=+2.0,
    demand_adj=+0.3,
    description="OPEC partially holds + demand surprise to the upside. Tightest realistic outcome.",
)

engine.add_scenario(
    "Bear Case (Oversupply)",
    opec_crude_adj=-0.5,
    demand_adj=-0.5,
    non_opec_adj=+0.3,
    description="Full OPEC unwind + weak demand + strong non-OPEC growth.",
)

results = engine.run_all()

# Print overview
print(f"Scenarios defined: {len(engine.scenarios)}")
print(f"Forecast months:   {engine.forecast_mask.sum()}")
print()
engine.print_summary()

In [ ]:
# === Quarterly Scenario Comparison ===
# Average balance (mb/d) under each scenario, by quarter.
# Positive = surplus, negative = deficit.

qtable = engine.summary_table()

q_styled = (
    qtable.style
    .format("{:+.2f}")
    .set_caption("Quarterly Balance by Scenario (mb/d)")
    .background_gradient(cmap="RdYlGn", axis=None, vmin=-3, vmax=3)
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th", "props": [("background-color", "#2c3e50"), ("color", "white"), ("padding", "8px")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "6px 10px")]},
    ])
)

display(q_styled)

In [ ]:
# === Scenario Balance Overlay Chart ===
fig = engine.plot_comparison(
    save_path=str(OUTPUT_DIR / "charts" / "scenario_comparison.png"),
)
plt.show()

In [ ]:
# === Scenario Interpretation ===
# What do these numbers actually mean for the market?

base_bal = results["Base Case (EIA STEO)"]["balance"]["balance"]
opec_bal = results["OPEC Holds Cuts"]["balance"]["balance"]
bull_bal = results["Bull Case (Tight Market)"]["balance"]["balance"]
bear_bal = results["Bear Case (Oversupply)"]["balance"]["balance"]

# Q2 and Q3 2026 averages for each scenario
def q_avg(bal, q_start, q_end):
    mask = (bal.index >= q_start) & (bal.index <= q_end)
    return bal[mask].mean()

q2_start, q2_end = "2026-04", "2026-06"
q3_start, q3_end = "2026-07", "2026-09"

interpretation = f"""
### Scenario Takeaways

**Base Case (STEO as-is):** The EIA sees the market moving from a +2.7 mb/d surplus in Feb 2026
to a deficit in March, driven by their forecast of OPEC production dropping from 29.25 to 23.08 mb/d.
Q2 avg balance: {q_avg(base_bal, q2_start, q2_end):+.2f} mb/d. Q3: {q_avg(base_bal, q3_start, q3_end):+.2f} mb/d.

**OPEC Holds Cuts:** If OPEC maintains closer to current production levels (~27 mb/d) instead of
the aggressive EIA unwind, the Q2 balance shifts to {q_avg(opec_bal, q2_start, q2_end):+.2f} mb/d and
Q3 to {q_avg(opec_bal, q3_start, q3_end):+.2f} mb/d. This is probably the most realistic
scenario given recent OPEC+ rhetoric about maintaining cuts.

**Bull Case:** Under the tightest scenario (partial OPEC hold + demand surprise),
Q3 balance reaches {q_avg(bull_bal, q3_start, q3_end):+.2f} mb/d, which would imply
strong backwardation in the Brent curve.

**Bear Case:** Full unwind + demand disappointment pushes the balance to
{q_avg(bear_bal, q2_start, q2_end):+.2f} mb/d in Q2 and {q_avg(bear_bal, q3_start, q3_end):+.2f} mb/d in Q3,
consistent with contango.

**What to watch:** OPEC+ meeting decisions and production data are the swing factor.
The spread between the bull and bear case is roughly {q_avg(bull_bal, q3_start, q3_end) - q_avg(bear_bal, q3_start, q3_end):.1f} mb/d
in Q3 2026, which translates to a meaningful difference in term structure and outright prices.
"""

display(Markdown(interpretation))